# PricePoint — 07 · Gradient Boosting

**What it is:** builds small decision trees **one after another**. The first tree makes a rough guess. Each new tree is trained to fix the **mistakes the previous trees left behind**. The final price is the first guess plus all the small corrections added up.

**Not the same as Random Forest:** a Random Forest grows its trees independently and averages them. Gradient Boosting grows them in a chain, each one learning from the last.

**Settings:** the same as the final model in `src/train_and_export.py` (the one the website uses), so these scores describe the model that is actually deployed. `learning_rate` is how big each correction step is — small steps are safer.

In [ ]:
import json

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed, so charts will be skipped. To see them: pip install matplotlib")

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
# --- file paths -------------------------------------------------------
# works whether Jupyter was opened in ml/notebooks, in ml, or in the repo root

for base in (Path(".."), Path("."), Path("ml")):
    if (base / "data" / "laptop_price_clean.csv").exists():
        DATA_DIR = base / "data"
        MODEL_DIR = base / "models"
        break
else:
    raise FileNotFoundError(
        "Cannot find data/laptop_price_clean.csv - open this notebook from the ml folder or the ml/notebooks folder."
    )

MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
# --- load the clean data ---------------------------------------------

clean = pd.read_csv(DATA_DIR / "laptop_price_clean.csv")

# True / False -> 1 / 0 so the models can use them
clean["touchscreen"] = clean["touchscreen"].astype(int)
clean["ips"] = clean["ips"].astype(int)

# the columns the models will use
categorical_columns = ["brand", "type", "cpu_brand", "storage_type", "gpu_brand", "os"]
numeric_columns = [
    "inches",
    "touchscreen",
    "ips",
    "resolution_pixels",
    "cpu_speed_ghz",
    "ram_gb",
    "storage_gb",
    "weight_kg",
]

X = clean[categorical_columns + numeric_columns]
y = clean["price_bdt"]

print("rows:", len(X))
print("features:", X.shape[1])

## Split into training and testing

In [ ]:
# --- split into training and testing --------------------------------
# the model only ever learns from the training part,
# the test part stays hidden until we score it

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("training rows:", len(X_train))
print("testing rows :", len(X_test))

## Build the pipeline

Two jobs in one object:
1. turn the text columns into numbers (one-hot encoding)
2. fit the model on those numbers

Trees do not care about scale, so no scaler is needed.

In [ ]:
# this turns the 6 text columns into numbers:
# one extra 0/1 column per category (brand_Apple, brand_Dell, ...)
preprocessor = ColumnTransformer(
    transformers=[
        (
            "category",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
        ("number", "passthrough", numeric_columns),
    ]
)

In [ ]:
# 2 steps: turn text into numbers -> fit the model
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=120,     # number of small trees, built one after another
        max_depth=4,          # each tree is small
        learning_rate=0.08,   # how much of each tree's correction we keep
        subsample=0.85,       # each tree sees a random 85% of the laptops
        random_state=42,
    )),
])

In [ ]:
pipeline.fit(X_train, y_train)
print("model trained")

## Score it

Three numbers, same meaning everywhere in this project:

- **MAE** — average miss in taka ("on average we are off by X BDT")
- **RMSE** — like MAE, but big mistakes are punished harder
- **R²** — how much of the price spread the model explains (1.0 = perfect, 0 = useless)

We score twice: on the held-out **test set**, and with **5-fold cross-validation** (more reliable — the data is shuffled and scored 5 different ways).

In [ ]:
# --- score on the test set -------------------------------------------

y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
r2_train = pipeline.score(X_train, y_train)

print("R² on training data :", round(r2_train, 3))
print("R² on test data     :", round(r2, 3))
print("MAE                 :", round(mae), "BDT")
print("RMSE                :", round(rmse), "BDT")

## Cross-validation

Shuffle the data, train 5 times on different slices, average the scores.

In [ ]:
# --- 5-fold cross-validation on the full data -----------------------
# splits the data 5 ways, trains 5 times, averages the scores

cv_mae = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_mean_absolute_error")
cv_rmse = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_r2 = cross_val_score(pipeline, X, y, cv=5, scoring="r2")

print("CV MAE :", round(cv_mae.mean()), "±", round(cv_mae.std()))
print("CV RMSE:", round(cv_rmse.mean()), "±", round(cv_rmse.std()))
print("CV R²  :", round(cv_r2.mean(), 3), "±", round(cv_r2.std(), 3))

### Watch it improve as trees are added

After 1 tree the guess is rough. Every extra tree fixes some of the remaining error, so the error drops. This chart is the "each tree fixes the last one's mistakes" idea in one picture.

In [ ]:
# --- test error after 1, 2, 3 ... trees ------------------------------

X_test_ready = pipeline.named_steps["preprocessing"].transform(X_test)
booster = pipeline.named_steps["model"]

mae_by_trees = [
    mean_absolute_error(y_test, y_stage)
    for y_stage in booster.staged_predict(X_test_ready)
]

for trees in (1, 10, 30, 60, 120):
    print(f"after {trees:>3} trees: MAE = {round(mae_by_trees[trees - 1]):>6} BDT")

if plt is not None:
    plt.figure(figsize=(7, 3.5))
    plt.plot(range(1, len(mae_by_trees) + 1), mae_by_trees)
    plt.xlabel("number of trees")
    plt.ylabel("test MAE (BDT) — lower is better")
    plt.title("Gradient Boosting: error falls as trees are added")
    plt.tight_layout()
    plt.show()

### Which features did it rely on most?

In [ ]:
# which features does this model rely on most?

feature_names = pipeline.named_steps["preprocessing"].get_feature_names_out()
importances = pd.Series(
    pipeline.named_steps["model"].feature_importances_, index=feature_names
)

print(importances.sort_values(ascending=False).head(10).to_string())

## Try it on 3 real laptops

In [ ]:
# --- try it on 3 real laptops from the test set ---------------------

sample = X_test.head(3)
sample_predicted = pipeline.predict(sample)

for position in range(3):
    real_price = y_test.iloc[position]
    predicted_price = round(sample_predicted[position])
    print("real:", real_price, "BDT    predicted:", predicted_price, "BDT")

## Save the scores

Notebook `08_compare_models.ipynb` reads these files.

In [ ]:
# --- save the scores (notebook 08 reads these files) ----------------

metrics = {
    "model": "Gradient Boosting",
    "test_mae": round(mae, 2),
    "test_rmse": round(rmse, 2),
    "test_r2": round(r2, 4),
    "cv_mae": round(cv_mae.mean(), 2),
    "cv_rmse": round(cv_rmse.mean(), 2),
    "cv_r2": round(cv_r2.mean(), 4),
}

metrics_path = MODEL_DIR / "metrics_gradient_boosting.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("saved:", metrics_path)

**Next notebook:** `08_compare_models.ipynb` — puts all six side by side.